This notebook shows the first attempt of importing the flight delay data

## Import Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

Data is taken from this page: https://transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ&QO_fu146_anzr=b0-gvzr

With the following columns selected:
* Year
* Quarter
* Month
* DayofMonth
* DayOfWeek
* FlightDate
* Reporting_Airline
* Flight_Number_Reporting_Airline
* OriginAirportID
* Origin
* DestAirportID
* Dest
* ArrDelayMinutes
* ArrDel15

## Import Data 

In [2]:
df = pd.read_csv("../data/raw/flights_2026_01")
df.shape

(544003, 15)

In [3]:
df = df.sample(100_000)

In [4]:
df.shape

(100000, 15)

In [5]:
df.head()

,Unnamed: 0,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,ARR_DELAY_NEW,ARR_DEL15
176720,176720,2026,1,1,10,6,1/10/2026 12:00:00 AM,UA,496,13204,MCO,11618,EWR,0.0,0.0
542502,542502,2026,1,1,31,6,1/31/2026 12:00:00 AM,WN,4263,11292,DEN,13871,OMA,0.0,0.0
484206,484206,2026,1,1,28,3,1/28/2026 12:00:00 AM,OO,3843,11977,GRB,13487,MSP,0.0,0.0
497658,497658,2026,1,1,29,4,1/29/2026 12:00:00 AM,DL,2654,10397,ATL,13487,MSP,44.0,1.0
155073,155073,2026,1,1,9,5,1/9/2026 12:00:00 AM,MQ,3679,10821,BWI,13930,ORD,31.0,1.0


In [6]:
df.columns

Index(['Unnamed: 0', 'YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK',
       'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM',
       'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST',
       'ARR_DELAY_NEW', 'ARR_DEL15'],
      dtype='str')

In [7]:
df = df.drop("Unnamed: 0", axis=1) # drop index column

In [8]:
df.isna().sum()

YEAR                    0
QUARTER                 0
MONTH                   0
DAY_OF_MONTH            0
DAY_OF_WEEK             0
FL_DATE                 0
OP_UNIQUE_CARRIER       0
OP_CARRIER_FL_NUM       0
ORIGIN_AIRPORT_ID       0
ORIGIN                  0
DEST_AIRPORT_ID         0
DEST                    0
ARR_DELAY_NEW        5005
ARR_DEL15            5005
dtype: int64

* Only missing columns in this case are the dependant variable so we have to drop

In [9]:
df[['ARR_DELAY_NEW','ARR_DEL15']].dropna(inplace=True) # drop all missing dependant rows

In [10]:
df.dtypes

YEAR                   int64
QUARTER                int64
MONTH                  int64
DAY_OF_MONTH           int64
DAY_OF_WEEK            int64
FL_DATE                  str
OP_UNIQUE_CARRIER        str
OP_CARRIER_FL_NUM      int64
ORIGIN_AIRPORT_ID      int64
ORIGIN                   str
DEST_AIRPORT_ID        int64
DEST                     str
ARR_DELAY_NEW        float64
ARR_DEL15            float64
dtype: object

* Most datatypes make sense except FL_DATE should be datetime
* ARR_DEL15 should be binary

In [11]:
df['FL_DATE'].sample(10)

533632    1/31/2026 12:00:00 AM
505993    1/29/2026 12:00:00 AM
41614      1/3/2026 12:00:00 AM
480545    1/28/2026 12:00:00 AM
27106      1/2/2026 12:00:00 AM
246865    1/14/2026 12:00:00 AM
408935    1/24/2026 12:00:00 AM
232382    1/13/2026 12:00:00 AM
319103    1/18/2026 12:00:00 AM
282464    1/16/2026 12:00:00 AM
Name: FL_DATE, dtype: str

* FL_DATE does not contain scheduled departure time, we need to get the CRSDepTime column

In [12]:
pd.to_datetime(df['FL_DATE'])

/var/folders/5l/ms2zw9fx1x5f64wc8dg5fpp80000gn/T/ipykernel_65890/3494662676.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(df['FL_DATE'])


176720   2026-01-10
542502   2026-01-31
484206   2026-01-28
497658   2026-01-29
155073   2026-01-09
            ...    
513527   2026-01-30
216001   2026-01-12
136409   2026-01-08
453316   2026-01-26
247497   2026-01-14
Name: FL_DATE, Length: 100000, dtype: datetime64[us]

* May not require FL_Date due to already have the other date information

In [13]:
df['OP_UNIQUE_CARRIER'].value_counts()

OP_UNIQUE_CARRIER
WN    19201
DL    14560
AA    14162
OO    12179
UA    11702
YX     5449
AS     4734
MQ     4453
OH     3629
B6     3364
F9     2766
NK     2118
G4     1683
Name: count, dtype: int64

* It would be nice to use a lookup table to easily understand which airline is which

In [14]:
df['OP_CARRIER_FL_NUM'].nunique()

6202

* There are many flight numbers which might not be helpful in predictions, perhaps they can be combined with the airline to identify troublesome routes, though the combination of airline, origin, destination might do that already
* Probably safe to drop this column for now

In [15]:
df['ORIGIN_AIRPORT_ID'].nunique(), df['ORIGIN'].nunique()

(341, 341)

These columns convey the same information so one of them is unnecessary, probably can drop the ORIGIN because it is recommended to use ORIGIN_AIRPORT_ID for cross year analysis

Same situation with DEST_AIRPORT_ID and DEST

In [17]:
df.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN',
       'DEST_AIRPORT_ID', 'DEST', 'ARR_DELAY_NEW', 'ARR_DEL15'],
      dtype='str')

In [20]:
df[['ARR_DELAY_NEW','ARR_DEL15']].sample(5)

,ARR_DELAY_NEW,ARR_DEL15
391678,44.0,1.0
3713,29.0,1.0
399405,0.0,0.0
138716,37.0,1.0
374908,0.0,0.0


* Have to decide between a classification task and a regression task
* Either predict if flight will land more than fifteen minutes late or predict exactly how many minutes delayed (if at all it will land)

## Final Thoughts

* This dataset is fairly simple and does not require much preprocessing
* Will need to reimport import the dataset with a better choice of columns, it would be good to use a lookup table to convert all the Airports and Carriers into a recognizable format
* Will need to make a pipeline which says what to do with missing variables considering how the user will input the flight
* Stil missing some important information related to flight delays - weather
* Need to find a simple heuristic to use as a benchline, e.g. propoprtion of all flights from airline on a particular route which are delayed
* Will need a full year of data to account for seasonal effects